<a href="https://colab.research.google.com/github/marviiiin/Reproducibility-An-Image-is-worth-16-x16-words/blob/main/DL4CV_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm

In [ ]:
# Load the CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train, y_train = x_train[:5000]/255., y_train[:5000]
x_train = np.transpose(x_train, (0, 3, 1, 2))
x_test, y_test = x_test[:1000]/255., y_test[:1000]
x_test = np.transpose(x_test, (0, 3, 1, 2))

print(x_train.shape, x_train.dtype)
print(y_train.shape, y_train.dtype)
print(x_test.shape, x_test.dtype)
print(y_test.shape, y_test.dtype)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step
(5000, 3, 32, 32) float64
(5000, 1) uint8
(1000, 3, 32, 32) float64
(1000, 1) uint8


In [ ]:
class CIFAR(torch.utils.data.Dataset):
    def __init__(self, x, y, mode):
        self.x = x.astype(np.float32)
        self.y = y.astype(np.int64)
        self.mode = mode

        self.t_train = transforms.Compose([
            torchvision.transforms.Resize((224,224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.t_not_train = transforms.Compose([
            torchvision.transforms.Resize((224,224)),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.x[idx])
        if self.mode == 'train':
          img = self.t_train(img)
        else:
          img = self.t_not_train(img)

        return img, self.y[idx]

train_dataset = CIFAR(x_train[:4000], y_train[:4000], 'train')
val_dataset = CIFAR(x_train[4000:], y_train[4000:], 'val')
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=8, num_workers=4, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)

In [ ]:
model = torchvision.models.vit_b_16(weights='DEFAULT')
model.heads.head = nn.Linear(768, 10)
model = model.cuda()
#model.eval()

"""
for x, y in train_dataloader:
    with torch.no_grad():
      y_ = model(x.cuda()).detach().cpu()
    y_ = torch.argmax(y_, dim=1)

    print(y, y_)

    break
"""

model.train()

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [ ]:
# compute accuracy
def acc(train_dataloader, model):
  model.eval()

  total = 0
  correct = 0
  for x, y in train_dataloader:
    with torch.no_grad():
      y_ = model(x.cuda()).detach().cpu()
    y_ = torch.argmax(y_, dim=1)

    correct += torch.sum((y_ == y[:,0]).float())
    total += len(x)

  return (correct/total).item()

In [ ]:
optimizer1 = optim.Adam([p for n,p in model.named_parameters() if n.startswith('heads.head.')], lr=0.001)
optimizer2 = optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

best_acc = 0.0
for i in range(100):
  if i < 10:
    optimizer = optimizer1
  else:
    optimizer = optimizer2

  model.train()
  for x, y in tqdm(train_dataloader):
    # zero the parameter gradients
    optimizer.zero_grad()

    # forward + backward + optimize
    outputs = model(x.cuda())
    loss = criterion(outputs, y[:,0].cuda())
    loss.backward()
    optimizer.step()

  val_acc = acc(val_dataloader, model)
  print(i, loss.item(), acc(train_dataloader, model), val_acc)

  if val_acc > best_acc:
    print('saved at', i)
    torch.save(model.state_dict(), 'mymodel.pth')
    best_acc = val_acc

print('Finished Training')

100%|██████████| 500/500 [01:05<00:00,  7.66it/s]


0 0.11394166946411133 0.9559999704360962 0.9309999942779541
saved at 0


100%|██████████| 500/500 [01:05<00:00,  7.68it/s]


1 0.016200555488467216 0.9679999947547913 0.9259999990463257


100%|██████████| 500/500 [01:05<00:00,  7.68it/s]


...

43 1.6124475002288818 0.5232499837875366 0.49300000071525574


100%|██████████| 500/500 [01:14<00:00,  6.74it/s]


44 1.2287051677703857 0.5367500185966492 0.48100000619888306


 87%|████████▋ | 436/500 [01:04<00:09,  6.74it/s]


KeyboardInterrupt: 